<a href="https://colab.research.google.com/github/NYChase/IBMDataScienceProjects/blob/Data-Visualization-with-Python/Dash_Components.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dash Components

Objectives
After completing the lab you will be able to:

*   Know how to add multiple graphs to the dashboard
*   Work with Dash Callbacks to handle multiple outputs

Estimated time needed: 30 minutes

To do:
* Design layout for the application.
* Create a callback function. Add callback decorator, define inputs and outputs.
* Review the helper function that performs computation on the provided inputs.
* Create 5 line graphs.
* Run the application.

In [2]:
!pip install jupyter-dash plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.6 MB/s eta 0:00:00


In [3]:
from jupyter_dash import JupyterDash
from dash import dcc, html
import plotly.express as px

In [5]:
# Import required libraries
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the airline data into pandas dataframe
airline_data =  pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
                            encoding = "ISO-8859-1",
                            dtype={'Div1Airport': str, 'Div1TailNum': str,
                                   'Div2Airport': str, 'Div2TailNum': str})

# Create a dash application
app = dash.Dash(__name__)

# Build dash app layout
app.layout = html.Div(children=[ html.H1('Flight Delay Time Statistics',
                                style={'textAlign': 'center', 'color': '#503D36',
                                'font-size': 30}),
                                html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                type='number', style={'height':'35px', 'font-size': 30}),],
                                style={'font-size': 30}),
                                html.Br(),
                                html.Br(),
                                # Segment 1
                                html.Div([
                                        html.Div(dcc.Graph(id='carrier-plot')),
                                        html.Div(dcc.Graph(id='weather-plot'))
                                ], style={'display': 'flex'}),
                                # Segment 2
                                html.Div([
                                        html.Div(dcc.Graph(id='nas-plot')),
                                        html.Div(dcc.Graph(id='security-plot'))
                                ], style={'display': 'flex'}),
                                # Segment 3
                                html.Div(dcc.Graph(id='late-plot'), style={'width':'65%'})
                                ])

""" Compute_info function description

This function takes in airline data and selected year as an input and performs computation for creating charts and plots.

Arguments:
    airline_data: Input airline data.
    entered_year: Input year for which computation needs to be performed.

Returns:
    Computed average dataframes for carrier delay, weather delay, NAS delay, security delay, and late aircraft delay.

"""
def compute_info(airline_data, entered_year):
    # Select data
    df =  airline_data[airline_data['Year']==int(entered_year)]
    # Compute delay averages
    avg_car = df.groupby(['Month','Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month','Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month','Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month','Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month','Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late

"""Callback Function

Function that returns fugures using the provided input year.

Arguments:

    entered_year: Input year provided by the user.

Returns:

    List of figures computed using the provided helper function `compute_info`.
"""
# Callback decorator
@app.callback( [
               Output(component_id='carrier-plot', component_property='figure'),
               Output(component_id='weather-plot', component_property='figure'),
               Output(component_id='nas-plot', component_property='figure'),
               Output(component_id='security-plot', component_property='figure'),
               Output(component_id='late-plot', component_property='figure')
               ],
               Input(component_id='input-year', component_property='value'))
# Computation to callback function and return graph
def get_graph(entered_year):

    # Compute required information for creating graph from the data
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)

    # Line plot for carrier delay
    carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline', title='Average carrrier delay time (minutes) by airline')
    # Line plot for weather delay
    weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline', title='Average weather delay time (minutes) by airline')
    # Line plot for nas delay
    nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline', title='Average NAS delay time (minutes) by airline')
    # Line plot for security delay
    sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline', title='Average security delay time (minutes) by airline')
    # Line plot for late aircraft delay
    late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline', title='Average late aircraft delay time (minutes) by airline')

    return[carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

# Run the app
if __name__ == '__main__':
    app.run(mode='inline')

<IPython.core.display.Javascript object>

# NYChase: What is the value of using Dash to create dashboards versus other data visualization tools that I've already used?
That's a great question! You've likely used tools like Matplotlib, Seaborn, or Plotly for creating static or interactive plots. Dash builds upon libraries like Plotly but offers a different paradigm, especially for building full-fledged web applications.

Here's a breakdown of the value of using Dash compared to other tools you may have used:

* Interactive Web Applications: The key differentiator is that Dash allows you to build entire interactive web dashboards directly in Python, without needing to write HTML, CSS, or JavaScript. This is a significant advantage if you want to share your visualizations with others as a standalone application.
* Callbacks for Interactivity: Dash's callback mechanism (as seen in the code) is a powerful way to connect user inputs (like selecting a year) to updates in the output (like changing the plots). This makes it relatively straightforward to create dynamic and responsive dashboards. While other libraries might offer some interactivity within a notebook or static HTML export, Dash is designed for building more complex, multi-component interactions in a web environment.
* Component-Based Layout: Dash uses a component-based approach for building the layout of your dashboard. You use Python classes that represent HTML elements (html.Div, html.H1, etc.) and Dash Core Components (dcc.Graph, dcc.Input, etc.). This structured approach can make it easier to organize and manage the different parts of your dashboard.
* Integration with Python Ecosystem: Since Dash is a Python framework, it seamlessly integrates with your existing Python data analysis and processing workflows. You can use libraries like pandas, NumPy, and scikit-learn to process your data and then use Dash to visualize the results.
* Scalability: While simple dashboards are easy to create, Dash can also be used to build more complex applications. You can deploy Dash applications to various platforms, making them accessible to a wider audience.

**Compared to libraries like Matplotlib or Seaborn**: These are excellent for creating static plots and visualizations within a notebook or for generating images. However, they are not designed for building interactive web applications with multiple connected components and user inputs in the way Dash is.

**Compared to Plotly (used independently)**: Plotly itself can create interactive plots that can be embedded in web pages or notebooks. Dash uses Plotly for generating the figures, but it provides the framework for building the overall dashboard structure, handling user interactions through callbacks, and deploying the application as a web service.

In summary, if your goal is to create interactive web-based dashboards with multiple linked visualizations and user controls, Dash provides a more integrated and streamlined approach within the Python ecosystem compared to using visualization libraries alone. If you primarily need static plots or simple interactive plots within a notebook, other libraries might be sufficient.